# Timing Analysis

Picks up from **baseline-analysis.ipynb**. Loads the `noise_filtered` waveforms
and pre-computed baselines, then runs the timing pipeline:

1. **Find Peaks** channel min/max amplitudes per event
2. **Calculate Thresholds** CFD threshold per channel (% of peak)
3. **Threshold Crossings** sample index where waveform crosses threshold
4. **Events of Interest** events where both trigger and signal fired
5. **Trigger-to-Signal Timings** from trigger crossing to each signal crossing
6. **Signal-to-Signal Timings** between every channel pair

## Notes

Datasets collected at Berkeley EOS.

- TRBHM: [2025-10-22 10-03-47.951971](https://drive.google.com/drive/folders/1cN6eJw0Rpp8GpyTnQ9kpw3sB65hqh14h?usp=drive_link) (1408 events)
- ASOC: [2025-10-22 11-32-56.502022](https://drive.google.com/drive/folders/16gO0CscjxJqfUVxIQYBA4Vx00oQiK_yD?usp=drive_link) (10404 events)

Link to Whisper Data: https://drive.google.com/drive/folders/1QALq6wFnaYW2QnPZEMOzGRIXo8y3_oj0?usp=drive_link

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib ipympl

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

from naludaq.board import Board

from toolbox.filters.analysis_pipeline import AnalysisPipeline
from toolbox.filters.steps import (
    peaks_step, thresholds_step, threshold_x_step,
    events_of_interest_step, trig_to_signal_step,
    channels_with_events_step, channel_pairings_step,
    signal_to_signal_step, signals_before_trig_step,
)
from toolbox.plotting.save_plots import save_plot
from toolbox.plotting.interactive_plot import interactive_plot

## Configuration

| Variable | Description |
|---|---|
| `BOARD` | `"trbhm"` or `"asoc"` â€” must match baseline-analysis |
| `BASELINE_RESULTS_DIR` | Folder written by baseline-analysis Save Results cell |
| `CHUNK_SIZE` | Events per chunk (`None` = all at once) |
| `EDGE_CONFIG` | Per-channel edge direction â€” `"falling"` or `"rising"` |
| `PERCENTAGE_THRESHOLD` | CFD fraction per channel (0â€“1) |
| `FALLING_THRESHOLD` | Hard minimum ADC for a falling-edge crossing |
| `RISING_THRESHOLD` | Hard minimum ADC for a rising-edge crossing |
| `TIMING_BIN_SIZE` | Histogram bin width in samples |
| `TIMING_WINDOW` | Plot range `(-TIMING_WINDOW, +TIMING_WINDOW)` in samples |

In [ ]:
BOARD = "trbhm"  # "trbhm" or "asoc"

BASELINE_RESULTS_DIR = Path("data") / "results"
CHECKPOINT_FILE      = f"{BOARD}-baseline"

CHUNK_SIZE = None

# Per-channel edge direction (ch 7 = trigger)
# TRBHM : signals fall, trigger rises
# ASOC  : signals fall, trigger rises  (adjust if needed)
EDGE_CONFIG = np.array(
    ["falling", "falling", "falling", "falling",
     "falling", "falling", "falling", "rising"]
)

# CFD fraction per channel
PERCENTAGE_THRESHOLD = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.4])

# Hard amplitude guards (ADC counts)
FALLING_THRESHOLD = -400   # signal channels must go below this
RISING_THRESHOLD  =  100   # trigger channel must go above this

TIMING_BIN_SIZE = 3   # samples
TIMING_WINDOW   = 200  # samples n for timing window

# Derived
CHANNELS        = list(range(8))
SIGNAL_CHANNELS = list(range(7))
TRIGGER_CHANNEL = 7

board = Board("trbhm") if BOARD == "trbhm" else Board("aodsoc_asoc")
SAMPLING_RATE = board.params["samplingrate"] * 1e9

PLOTS_DIR = Path("plots") / "timing"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print("Board Summary")
print("-------------")
print(f"Model:         {board.params['model']}")
print(f"Channels:      {board.params['channels']}")
print(f"Windows:       {board.params['windows']}")
print(f"Samples:       {board.params['samples']}")
print(f"Sampling Rate: {board.params['samplingrate']} GHz")

## Load Baseline Results

Loads outputs saved by **baseline-analysis.ipynb**. The key input is
`noise_filtered` (CW-notch-filtered waveforms) and `baselines_0_window`
(first-window baseline per channel per event), which the threshold step uses.

In [ ]:
def load_results(results_dir: Path, prefix: str) -> dict:
    out = {}
    for f in sorted(results_dir.glob(f"{prefix}-*.npy")):
        key = f.stem.replace(f"{prefix}-", "")
        try:
            out[key] = np.load(f, mmap_mode="r", allow_pickle=True)
            print(f"Loaded  {key:45s}  {out[key].shape}")
        except Exception:
            out[key] = np.load(f, allow_pickle=True)
            print(f"Loaded  {key:45s}  (pickle)")
    return out

baseline = load_results(BASELINE_RESULTS_DIR, CHECKPOINT_FILE)

noise_filtered = baseline["noise_filtered"]    # (N, C, S)
baselines_0    = baseline["baselines_0_window"] # (C, N)
signal_events  = baseline["signal_events"]
noise_events   = baseline["noise_events"]

N, C, S = noise_filtered.shape
print(f"\nData shape: {noise_filtered.shape}  (events, channels, samples)")

## Preview

In [ ]:
interactive_plot(noise_filtered, title=f"{BOARD.upper()} noise_filtered")

## Timing Pipeline

The first-window baseline step is included so `thresholds_step` can read
`baselines_0_window` from the shared results dict.  The remaining steps:

| Step | What it does |
|---|---|
| `peaks_step` | Min/max amplitude per channel per event |
| `thresholds_step` | CFD level = baseline + fraction Ã— (peak âˆ’ baseline) |
| `threshold_x_step` | Sample index of threshold crossing |
| `events_of_interest_step` | Events where trigger AND â‰¥1 signal fired |
| `trig_to_signal_step` | Î”n from trigger crossing to each signal crossing |
| `signal_to_signal_step` | Î”n between every channel pair |

In [ ]:
pipe = AnalysisPipeline()

# Re-run first-window baseline so thresholds_step can read it from results
from toolbox.filters.steps import baseline_step as _baseline_step
pipe.add(_baseline_step,
         source_key=None, method=2, window_len=64, window_index=0, axis=2)

pipe.add(peaks_step)

pipe.add(
    thresholds_step,
    percentage_threshold=PERCENTAGE_THRESHOLD,
    baselines="baselines_0_window",
    channel_min_peaks="channel_min_peaks",
    channel_max_peaks="channel_max_peaks",
    edge=EDGE_CONFIG,
)

pipe.add(
    threshold_x_step,
    edge=EDGE_CONFIG,
    falling_threshold=FALLING_THRESHOLD,
    rising_threshold=RISING_THRESHOLD,
)

pipe.add(events_of_interest_step)
pipe.add(trig_to_signal_step)
pipe.add(signal_to_signal_step, filter_sig_before_trig=False)

print(pipe)

## Run Pipeline

In [ ]:
timing_results = pipe.run(noise_filtered, chunk_size=CHUNK_SIZE)

print("\nResult keys:")
for k, v in timing_results.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:45s}  {str(v.shape)}")
    else:
        print(f"  {k:45s}  {type(v).__name__}")

## Save Timing Results

Saves numpy-serialisable keys.  Dict-valued keys (`sig2sig_timings`, etc.) are
saved with `allow_pickle=True`.

In [ ]:
TIMING_RESULTS_DIR = Path("data") / "results"
TIMING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REPLACE = True

skip_keys = set()  # add keys here to exclude from saving

for key, val in timing_results.items():
    if key in skip_keys:
        continue
    out = TIMING_RESULTS_DIR / f"{BOARD}-timing-{key}.npy"
    if REPLACE or not out.exists():
        np.save(out, np.asarray(val) if isinstance(val, np.ndarray) else val,
                allow_pickle=True)
        print(f"Saved  {key}")

---
## Trigger-to-Signal Timings

`trig2sig_delta_n[i]` is the sample offset from the trigger crossing to the
signal crossing for the i-th (event, channel) pair in `trig2sig_event_indices`.
Negative values mean the signal crossed *before* the trigger.

In [ ]:
delta_n      = np.asarray(timing_results["trig2sig_delta_n"])          # (M,)
event_idx    = np.asarray(timing_results["trig2sig_event_indices"])     # (M,)
channel_idx  = np.asarray(timing_results["trig2sig_channel_indices"])   # (M,)

print(f"Trigger-to-signal pairs: {len(delta_n)}")
print(f"Unique events:           {np.unique(event_idx).size}")
print(f"Delta-n range:           [{delta_n.min():.0f}, {delta_n.max():.0f}] samples")
print(f"Negative (before trig):  {(delta_n < 0).sum()}")

In [ ]:
BINS = np.arange(-TIMING_WINDOW, TIMING_WINDOW + TIMING_BIN_SIZE, TIMING_BIN_SIZE)

fig, axes = plt.subplots(2, 4, figsize=(15, 8), sharey=False)
axes = axes.ravel()


for ax_idx, ch in enumerate(SIGNAL_CHANNELS):
    mask = channel_idx == ch
    vals = delta_n[mask]
    finite = vals[np.isfinite(vals)]

    ax = axes[ax_idx]
    ax.hist(finite, bins=BINS)
    ax.set_title(f"Ch {ch}")
    ax.set_xlabel("Trigger-to-Signal Timing [n] (samples)")
    ax.set_ylabel("Counts")
    ax.set_yscale("log")
    ax.axvline(0, color="red", linestyle="--", linewidth=1, label="trigger")
    if len(finite):
        ax.set_title(f"Ch {ch}  (n={len(finite)}, median={np.median(finite):.1f})")

for ax in axes[len(SIGNAL_CHANNELS):]:
    ax.set_visible(False)

fig.suptitle(f"{BOARD.upper()} Trigger-to-Signal Timing  (bin={TIMING_BIN_SIZE} samples)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_plot(fig, PLOTS_DIR / f"{BOARD}-trig2sig")
plt.show()

In [ ]:
neg_mask    = delta_n < 0
neg_events  = event_idx[neg_mask]
neg_channels = channel_idx[neg_mask]
print(f"Events with signal before trigger: {np.unique(neg_events).size}")
print(f"Event indices: {np.unique(neg_events)[:20]}{'...' if len(np.unique(neg_events))>20 else ''}")

### Trigger Jitter

In [ ]:
trig_crossings = timing_results["threshold_x"][:, 7]          # (N,)
finite = trig_crossings[np.isfinite(trig_crossings)]

fig = plt.figure()
plt.title(f"{BOARD.upper()} - Trigger Crossing Index (samples)")
plt.hist(finite, bins=128)
plt.xlabel("Sample Index")
plt.ylabel("Counts")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-trigger-crossing")
plt.show()


---
## Signal-to-Signal Timings

`sig2sig_timings` is a dict keyed by `"i-j"` channel pair strings.  Each value
is an array of Î”n values (channel i crossing minus channel j crossing) for
events where both channels fired.

In [ ]:
channel_pairs   = timing_results["sig2sig_channel_pairs"]
sig2sig_timings = timing_results["sig2sig_timings"].item() \
    if isinstance(timing_results["sig2sig_timings"], np.ndarray) \
    else timing_results["sig2sig_timings"]

# Exclude pairs that involve the trigger channel
valid_pairs = [p for p in channel_pairs if TRIGGER_CHANNEL not in
               tuple(int(x) for x in p.split("-"))]
print(f"Total pairs: {len(channel_pairs)}  |  Signal-only pairs: {len(valid_pairs)}")

In [ ]:
def parse_pair(s):
    a, b = s.split("-")
    return int(a), int(b)

def pairs_for_channel(pairs, ch):
    return sorted([p for p in pairs if ch in parse_pair(p)],
                  key=parse_pair)

BINS = np.arange(-TIMING_WINDOW, TIMING_WINDOW + TIMING_BIN_SIZE, TIMING_BIN_SIZE)

for ch in SIGNAL_CHANNELS:
    ch_pairs = pairs_for_channel(valid_pairs, ch)
    if not ch_pairs:
        continue

    ncols = 3
    nrows = math.ceil(len(ch_pairs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows), sharey=False)
    axes = np.atleast_1d(axes).ravel()

    fig.suptitle(f"{BOARD.upper()} - Ch {ch} Signal-to-Signal  (bin={TIMING_BIN_SIZE} samples)")

    for ax, pair_key in zip(axes, ch_pairs):
        vals = np.asarray(sig2sig_timings[pair_key])
        vals = vals[np.isfinite(vals)]
        clipped = vals[(vals > -TIMING_WINDOW) & (vals < TIMING_WINDOW)]

        if len(clipped):
            median = np.median(clipped)
            mean   = np.mean(clipped)
            std    = np.std(clipped)
            label  = f"n={len(clipped)}  median={median:.1f}  mean={mean:.1f}  std={std:.1f}"
        else:
            label = "no data"

        ax.hist(clipped, bins=BINS, label=label)
        ax.set_title(f"Channels {pair_key}")
        ax.set_xlabel("n (samples)")
        ax.set_ylabel("Counts")
        ax.legend(fontsize=8)

    for ax in axes[len(ch_pairs):]:
        ax.set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    save_plot(fig, PLOTS_DIR / f"{BOARD}-sig2sig-ch{ch}")
    plt.show()

## Delay Matrix

Entry `[i, j]` is the **median** Î”n for the `i-j` channel pair restricted to
the Â±`TIMING_WINDOW` sample window.  Positive = channel i fires after channel j.
Off-diagonal asymmetry indicates a real propagation delay.

In [ ]:
n_sig = len(SIGNAL_CHANNELS)
delays = np.full((n_sig, n_sig), 0.0)
counts = np.zeros((n_sig, n_sig), dtype=int)

for pair_key in valid_pairs:
    i, j = parse_pair(pair_key)
    if i >= n_sig or j >= n_sig:
        continue
    vals = np.asarray(sig2sig_timings[pair_key])
    vals = vals[np.isfinite(vals)]
    clipped = vals[(vals > -TIMING_WINDOW) & (vals < TIMING_WINDOW)]
    if len(clipped):
        delays[i, j]  =  np.mean(clipped)
        delays[j, i]  = -np.mean(clipped)
        counts[i, j]  =  len(clipped)
        counts[j, i]  =  len(clipped)


In [ ]:
ordering_score = delays.mean(axis=1)
channel_order_simple = np.argsort(ordering_score)
print("Channel Order (Row-Mean):", channel_order_simple)
print("Scores:", ordering_score[channel_order_simple])


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(delays, cmap="RdBu_r", aspect="equal",
               vmin=-TIMING_WINDOW, vmax=TIMING_WINDOW)

ax.set_xticks(range(n_sig))
ax.set_yticks(range(n_sig))
ax.set_xticklabels([f"Ch {i}" for i in SIGNAL_CHANNELS])
ax.set_yticklabels([f"Ch {i}" for i in SIGNAL_CHANNELS])
ax.set_xlabel("Channel j")
ax.set_ylabel("Channel i")
ax.set_title(f"{BOARD.upper()} - Mean (samples)  [i-j, pos = i fires later]")

for i in range(n_sig):
    for j in range(n_sig):
        val = delays[i, j]
        txt = f"{val:.1f}" if np.isfinite(val) else ""
        ax.text(j, i, txt, ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, label="Mean (samples)")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-sig2sig-delay-matrix")
plt.show()

## Jitter Matrix

Entry `[i, j]` is the **standard deviation** of signal-to-signal timing and a measure of timing jitter
between channel pairs.

In [ ]:
jitter = np.full((n_sig, n_sig), np.nan)

for pair_key in valid_pairs:
    i, j = parse_pair(pair_key)
    if i >= n_sig or j >= n_sig:
        continue
    vals = np.asarray(sig2sig_timings[pair_key])
    vals = vals[np.isfinite(vals)]
    clipped = vals[(vals > -TIMING_WINDOW) & (vals < TIMING_WINDOW)]
    if len(clipped):
        jitter[i, j] = np.std(clipped)
        jitter[j, i] = np.std(clipped)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(jitter, cmap="viridis", aspect="equal")

ax.set_xticks(range(n_sig))
ax.set_yticks(range(n_sig))
ax.set_xticklabels([f"Ch {i}" for i in SIGNAL_CHANNELS])
ax.set_yticklabels([f"Ch {i}" for i in SIGNAL_CHANNELS])
ax.set_xlabel("Channel j")
ax.set_ylabel("Channel i")
ax.set_title(f"{BOARD.upper()} - Signal-to-Signal Timing Jitter (samples)")

for i in range(n_sig):
    for j in range(n_sig):
        val = jitter[i, j]
        txt = f"{val:.1f}" if np.isfinite(val) else ""
        ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="white")

plt.colorbar(im, ax=ax, label="Ïƒ (samples)")
plt.tight_layout()
save_plot(fig, PLOTS_DIR / f"{BOARD}-sig2sig-jitter-matrix")
plt.show()